# Lab 2 · Find the target with Grover's algorithm

**Qiskit Fall Fest 2026 @ UT San Antonio · Day 2 · Friday, Nov 20, 2026**

You'll build Grover's search **by hand**, make it work for **any** target, choose the **right number of iterations**, and finally run it on a **real IBM quantum computer** (or a realistic noisy copy).

| Part | You build | You find out |
|---|---|---|
| **A** | Grover on 2 qubits, gate by gate | why one iteration finds the target with 100% certainty |
| **B** | an oracle, a diffuser and `grover(w, k)` for any target | how the pieces generalize to $n$ qubits |
| **C** | the best number of iterations | why too many iterations **overshoot** |
| **D** | your circuit on a real chip | what noise does to Grover |
| **E** | *bonus:* two targets, and Qiskit's shortcut | amplitude amplification in general |

### How this notebook works

1. Write your code where it says `# YOUR CODE HERE`, run the cell, then run the `check(...)` cell under it.
2. It prints **correct! 🎉** or a hint. Stuck? Open the **💡 hint** box. Still stuck after 3 minutes? Skip and come back.
3. At the end, run `scoreboard()`, take a **screenshot** and post it in the event chat at the end of the lab.

Reminder from the slides: Qiskit writes bitstrings with **qubit 0 on the right**, so `'101'` means $q_2 = 1, q_1 = 0, q_0 = 1$.

---
## 0 · Setup

In [ ]:
YOUR_NAME = "Your Name"   # <- change this

In [ ]:
# %pip install --upgrade 'qiskit[visualization]' qiskit-aer qiskit-ibm-runtime -q   # only on your own laptop
from math import pi, sqrt, asin, sin, floor
import numpy as np
import qiskit
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, Operator
from qiskit.primitives import StatevectorSampler
from qiskit.visualization import plot_histogram

sampler = StatevectorSampler()
print("Qiskit", qiskit.__version__, "· ready!" if qiskit.__version__.startswith("2.") else "· please use Qiskit 2.x")

# Install and load the auto-checks for this lab
%pip install -q --no-deps --force-reinstall https://github.com/vantnprof/QFF2026_UTSA_grader/archive/refs/heads/main.zip
from qff_utsa_grader.lab2 import check, scoreboard

---
## Part A · Grover on 2 qubits, by hand

With $N = 4$ items and one marked item $w = 11$, a single Grover iteration lands **exactly** on the answer: $\sin\theta = \tfrac12$, so $\theta = 30^\circ$ and $\sin^2(3\theta) = 1$.

### A1 · Equal superposition

**Input:** a 2-qubit circuit `start2`. **Output:** $\lvert s\rangle = \tfrac12(\lvert00\rangle + \lvert01\rangle + \lvert10\rangle + \lvert11\rangle)$.

<details><summary><b>💡 Click for a hint</b></summary>

One Hadamard on **each** qubit: `start2.h([0, 1])`.

</details>

In [ ]:
start2 = QuantumCircuit(2)
# YOUR CODE HERE

Statevector(start2).draw("latex")

In [ ]:
check("A1")

### A2 · An oracle that marks $\lvert 11\rangle$

The oracle flips the sign of the marked state only: $O = \mathrm{diag}(1, 1, 1, -1)$.

**Input:** a 2-qubit circuit `oracle_11`. **Output:** exactly that matrix.

<details><summary><b>💡 Click for a hint</b></summary>

You met this matrix on Day 1: it's the **controlled-Z** gate, `qc.cz(0, 1)`.

</details>

In [ ]:
oracle_11 = QuantumCircuit(2)
# YOUR CODE HERE

print(Operator(oracle_11).data.real.round(3))

In [ ]:
check("A2")

### A3 · The 2-qubit diffuser

The diffuser reflects about $\lvert s\rangle$: $D = 2\lvert s\rangle\langle s\rvert - I$. The trick from the slides: in the **Hadamard basis**, it's a reflection about $\lvert00\rangle$, and $X\,\mathrm{CZ}\,X$ flips the sign of $\lvert00\rangle$ only.

**Input:** a 2-qubit circuit `diffuser2`. **Output:** $2\lvert s\rangle\langle s\rvert - I$ (an overall $-1$ is fine: it's a global phase).

<details><summary><b>💡 Click for a hint</b></summary>

Five layers: `h` on both, `x` on both, `cz(0, 1)`, `x` on both, `h` on both.

</details>

In [ ]:
diffuser2 = QuantumCircuit(2)
# YOUR CODE HERE

diffuser2.draw("mpl")

In [ ]:
check("A3")

### A4 · One Grover iteration

**Input:** `grover2` = `start2`, then `oracle_11`, then `diffuser2`. **Output:** the state $\lvert11\rangle$, with probability 1.

<details><summary><b>💡 Click for a hint</b></summary>

`grover2 = start2.compose(oracle_11).compose(diffuser2)`

</details>

In [ ]:
grover2 = None
# YOUR CODE HERE

if grover2 is not None:
    print(Statevector(grover2).probabilities_dict())

In [ ]:
check("A4")

🤔 Check it with *inversion about the mean*: after the oracle the amplitudes are $(\tfrac12, \tfrac12, \tfrac12, -\tfrac12)$. What is the mean? Where does each amplitude land?

---
## Part B · Any target, any size

### B1 · An oracle for any target $w$

**Input:** a bitstring `w` like `"101"`. **Output:** `make_oracle(w)`, an $n$-qubit circuit equal to $I - 2\lvert w\rangle\langle w\rvert$.

Recipe from the slides:
1. $X$ on every qubit where $w$ has a `0` (this moves $\lvert w\rangle$ to $\lvert 1\cdots1\rangle$)
2. a multi-controlled $Z$: `h` on the last qubit, `mcx(controls, last)`, `h` on the last qubit
3. undo step 1

<details><summary><b>💡 Click for a hint</b></summary>

Qubit $i$ corresponds to `w[-1 - i]`: use `enumerate(reversed(w))` to find the zeros.

`zeros = [i for i, b in enumerate(reversed(w)) if b == '0']`, then `if zeros: qc.x(zeros)`.

</details>

In [ ]:
def make_oracle(w):
    n = len(w)
    qc = QuantumCircuit(n)
    # YOUR CODE HERE
    return qc

make_oracle("101").draw("mpl")

In [ ]:
check("B1")

### B2 · A diffuser for any size

**Input:** a number of qubits `n`. **Output:** `make_diffuser(n)` equal to $2\lvert s\rangle\langle s\rvert - I$ (up to a global sign).

<details><summary><b>💡 Click for a hint</b></summary>

Same as A3, with a multi-controlled $Z$ instead of `cz`: `h`, `x` on all; `h(n-1)`, `mcx(list(range(n-1)), n-1)`, `h(n-1)`; `x`, `h` on all.

</details>

In [ ]:
def make_diffuser(n):
    qc = QuantumCircuit(n)
    # YOUR CODE HERE
    return qc

make_diffuser(3).draw("mpl")

In [ ]:
check("B2")

### B3 · Grover for any target

**Input:** a target `w` and a number of iterations `k`. **Output:** `grover(w, k)`: Hadamards on all qubits, then `k` times (oracle, then diffuser). No measurements.

<details><summary><b>💡 Stuck? Click for a hint</b></summary>

Start with `qc = QuantumCircuit(n)` and `qc.h(range(n))`.

In a loop: `qc.compose(make_oracle(w), inplace=True)` then `qc.compose(make_diffuser(n), inplace=True)`.

</details>

In [ ]:
def grover(w, k):
    n = len(w)
    qc = QuantumCircuit(n)
    # YOUR CODE HERE
    return qc

print(Statevector(grover("101", 2)).probabilities_dict())

In [ ]:
check("B3")

---
## Part C · How many iterations?

After $k$ iterations, $P_k = \sin^2\big((2k+1)\theta\big)$ with $\sin\theta = 1/\sqrt{N}$. The best $k$ makes $(2k+1)\theta$ closest to $\pi/2$.

### C1 · The best number of iterations

**Input:** a number of qubits `n`. **Output:** `best_k(n)`, the $k$ with the **highest** success probability.

<details><summary><b>💡 Click for a hint</b></summary>

`theta = asin(1 / sqrt(2 ** n))`, then `floor(pi / (4 * theta))`.

Check: `best_k(2)` = 1, `best_k(3)` = 2, `best_k(4)` = 3.

</details>

In [ ]:
def best_k(n):
    # YOUR CODE HERE
    return None

print([best_k(n) for n in range(2, 11)])

In [ ]:
check("C1")

### C2 · Success vs iterations

**Input:** `grover("101", k)` for $k = 0, 1, \ldots, 6$. **Output:** `p_success`, a list of 7 probabilities of finding `101`. Use the Sampler (1,000 shots is enough) or the exact `Statevector`.

<details><summary><b>💡 Click for a hint</b></summary>

`c = sampler.run([grover('101', k).measure_all(inplace=False)], shots=2000).result()[0].data.meas.get_counts()`, then `c.get('101', 0) / 2000`.

</details>

In [ ]:
p_success = []
# YOUR CODE HERE

if len(p_success) == 7:
    import matplotlib.pyplot as plt
    ks = np.arange(7); th = asin(1 / sqrt(8)); kk = np.linspace(0, 6.2, 300)
    plt.plot(kk, np.sin((2 * kk + 1) * th) ** 2, label="theory"); plt.plot(ks, p_success, "o", label="yours")
    plt.xlabel("iterations k"); plt.ylabel("P(find 101)"); plt.legend(); plt.show()

In [ ]:
check("C2")

🤔 Why does $P$ drop at $k = 3$, then climb again? (Think of the state as a vector **rotating** by $2\theta$ each step.)

---
## Part D · Run it on a real chip

Choose where to run:

* `USE_REAL_HARDWARE = False` (default): **FakeTorino**, a realistic noisy copy of the 133-qubit IBM Heron `ibm_torino`. Instant, no account needed.
* `USE_REAL_HARDWARE = True`: the **least busy real IBM QPU** on your account (saved in the morning session). Your job waits in a queue; the job ID is printed so you can fetch it later.

In [ ]:
USE_REAL_HARDWARE = False   # <- set to True to use a real IBM quantum computer
# Either using real or fake hardware is fine for achieving full points on the lab

if USE_REAL_HARDWARE:
    from qiskit_ibm_runtime import QiskitRuntimeService
    service = QiskitRuntimeService()
    backend = service.least_busy(operational=True, simulator=False)
else:
    from qiskit_ibm_runtime.fake_provider import FakeTorino
    backend = FakeTorino()
print(f"Backend: {backend.name} ({backend.num_qubits} qubits)")

### D1 · Transpile for the backend

**Input:** `grover("101", 2)` with measurements. **Output:** `isa`, the transpiled circuit (use `optimization_level=3`, it picks better qubits), and `cz_count`, its number of CZ gates.

<details><summary><b>💡 Click for a hint</b></summary>

`from qiskit.transpiler import generate_preset_pass_manager`

`pm = generate_preset_pass_manager(optimization_level=3, backend=backend)` then `isa = pm.run(grover('101', 2).measure_all(inplace=False))`

`cz_count = isa.count_ops().get('cz', 0)`

</details>

In [ ]:
isa = None
cz_count = None
# YOUR CODE HERE

if isa is not None:
    print("depth:", isa.depth(), "| gates:", dict(isa.count_ops()))

In [ ]:
check("D1")

### D2 · Run it

**Input:** your `isa` circuit, 2,000 shots, with the Runtime `Sampler`. **Output:** `hw_counts` (the counts dict) and `hw_success`, the fraction of shots that gave `101`.

On real hardware the job may wait in a queue: keep the printed job ID.

<details><summary><b>💡 Click for a hint</b></summary>

`from qiskit_ibm_runtime import SamplerV2 as Sampler`

`job = Sampler(mode=backend).run([isa], shots=2000)`, then `print(job.job_id())`

`hw_counts = job.result()[0].data.meas.get_counts()` and `hw_success = hw_counts.get('101', 0) / 2000`

</details>

In [ ]:
hw_counts = None
hw_success = None
# YOUR CODE HERE

if hw_counts:
    print(f"P(101) on {backend.name}: {hw_success:.3f}   (ideal: 0.945)")
    display(plot_histogram(hw_counts))

In [ ]:
check("D2")

🤔 Your circuit has `cz_count` CZ gates. With a CZ error of about 0.4%, what success rate would you **estimate** with $F \approx \prod_i(1 - \varepsilon_i)$? Is the measured value higher or lower, and why?

---
## Part E · Bonus

### E1 · Two marked items

Mark **both** `011` and `110` out of 8. With $M = 2$, $\sin\theta = \sqrt{2/8} = \tfrac12$, so **one** iteration finds one of them with certainty.

**Output:** `oracle_two` (flips both signs) and `grover_two` (Hadamards, then one oracle + diffuser).

<details><summary><b>💡 Click for a hint</b></summary>

Two sign flips in a row: `make_oracle('011').compose(make_oracle('110'))`.

</details>

In [ ]:
oracle_two = None
grover_two = None
# YOUR CODE HERE

if grover_two is not None:
    print(Statevector(grover_two).probabilities_dict())

In [ ]:
check("E1")

### E2 · Qiskit's shortcut

`qiskit.circuit.library.grover_operator(oracle)` builds one full Grover iteration (diffuser after oracle) for you.

**Output:** `lib_circuit`: Hadamards on 3 qubits, then `grover_operator(make_oracle("101"))` applied **twice**.

<details><summary><b>💡 Click for a hint</b></summary>

`from qiskit.circuit.library import grover_operator`; `G = grover_operator(make_oracle('101'))`

`lib_circuit = QuantumCircuit(3); lib_circuit.h(range(3)); lib_circuit.compose(G.power(2), inplace=True)`

</details>

In [ ]:
lib_circuit = None
# YOUR CODE HERE

if lib_circuit is not None:
    print(Statevector(lib_circuit).probabilities_dict())

In [ ]:
check("E2")

---
## 🏁 Scoreboard

Run this cell, take a **screenshot**, and post it in the event chat at the end of the lab.

In [ ]:
scoreboard()

---
*Qiskit Fall Fest 2026 @ UT San Antonio · Lab 2 by Van Tien Nguyen (UT San Antonio). Tested with Qiskit 2.6.2, qiskit-ibm-runtime 0.52.1, qiskit-aer 0.18.2.*